# MNIST Image Classification with a Restricted Boltzmann Machine and a Logistic Regression Classifier

Companion code for the report.
Trains an RBM on flattened MNIST images, then uses the RBM's hidden-layer features as input to a Logistic Regression classifier.
The notebook also includes a baseline Logistic Regression trained on raw pixels for comparison.


## Imports

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


## Load and preprocess MNIST

In [ ]:
# Load mnist from Keras
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

class_names = [
    "Zero", "One", "Two", "Three", "Four",
    "Five", "Six", "Seven", "Eight", "Nine"
]

# Normalize and reshape to flat vectors
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0
x_train = x_train.reshape(len(x_train), -1)
x_test  = x_test.reshape(len(x_test),  -1)
y_train = y_train.flatten()
y_test  = y_test.flatten()

# Batch the training data for the RBM
train_ds = (tf.data.Dataset.from_tensor_slices(x_train.astype(np.float32))
    .shuffle(5000, seed=42).batch(128).prefetch(tf.data.AUTOTUNE))


## RBM class definition

In [ ]:
class RBM(tf.Module):
    def __init__(self, n_visible, n_hidden, learning_rate=0.01, name=None):
        super().__init__(name=name)
        self.n_visible = n_visible
        self.n_hidden = n_hidden
        self.learning_rate = learning_rate
        initializer = tf.random.normal([n_visible, n_hidden], mean=0.0, stddev=0.01)
        self.W = tf.Variable(initializer, name="W")
        self.b_v = tf.Variable(tf.zeros([n_visible]), name="b_v")
        self.b_h = tf.Variable(tf.zeros([n_hidden]), name="b_h")

    def hidden_prob(self, v):
        return tf.nn.sigmoid(tf.matmul(v, self.W) + self.b_h)

    def visible_prob(self, h):
        return tf.nn.sigmoid(tf.matmul(h, tf.transpose(self.W)) + self.b_v)

    def sample_bernoulli(self, probs):
        random_uniform = tf.random.uniform(tf.shape(probs))
        return tf.cast(random_uniform < probs, tf.float32)

    def contrastive_divergence(self, v0):
        h0_prob = self.hidden_prob(v0)
        h0_sample = self.sample_bernoulli(h0_prob)
        v1_prob = self.visible_prob(h0_sample)
        v1_sample = self.sample_bernoulli(v1_prob)
        h1_prob = self.hidden_prob(v1_sample)

        batch_size = tf.cast(tf.shape(v0)[0], tf.float32)
        positive_grad = tf.matmul(tf.transpose(v0), h0_prob)
        negative_grad = tf.matmul(tf.transpose(v1_sample), h1_prob)

        dW = (positive_grad - negative_grad) / batch_size
        db_v = tf.reduce_mean(v0 - v1_sample, axis=0)
        db_h = tf.reduce_mean(h0_prob - h1_prob, axis=0)

        self.W.assign_add(self.learning_rate * dW)
        self.b_v.assign_add(self.learning_rate * db_v)
        self.b_h.assign_add(self.learning_rate * db_h)

        recon_error = tf.reduce_mean(tf.square(v0 - v1_prob))
        return recon_error

    def reconstruct(self, v):
        h_prob = self.hidden_prob(v)
        v_prob = self.visible_prob(h_prob)
        return v_prob


## Train the RBM

In [ ]:
# Set up RBM with the input shape that matches the data
n_visible = 784
n_hidden = 128
learning_rate = 0.05
epochs = 15

rbm = RBM(n_visible=n_visible, n_hidden=n_hidden, learning_rate=learning_rate)

# Train for `epochs` epochs
history = []
for epoch in range(1, epochs + 1):
    epoch_errors = []
    for batch in train_ds:
        err = rbm.contrastive_divergence(batch)
        epoch_errors.append(float(err.numpy()))
    mean_err = np.mean(epoch_errors)
    history.append(mean_err)
    print(f"Epoch {epoch:02d}/{epochs} - reconstruction loss: {mean_err:.6f}")


## Logistic Regression on RBM features

In [ ]:
# Get hidden representations from the trained RBM
def get_hidden_features(rbm, x):
    h = rbm.hidden_prob(tf.constant(x.astype(np.float32)))
    return h.numpy()

X_train_feat = get_hidden_features(rbm, x_train)
X_test_feat = get_hidden_features(rbm, x_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_feat, y_train.flatten())
y_pred = clf.predict(X_test_feat)
print("Accuracy (RBM features):", accuracy_score(y_test, y_pred))


## Confusion matrix (RBM features)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = [class_names[i] for i in range(10)]

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", values_format="d")
plt.title("MNIST Confusion Matrix - RBM features")
plt.show()


## Baseline: Logistic Regression on raw pixels

Same procedure, but the classifier is trained directly on the flattened pixel values rather than the RBM's hidden features.


In [ ]:
clf_raw = LogisticRegression(max_iter=1000)
clf_raw.fit(x_train, y_train)
y_pred_raw = clf_raw.predict(x_test)
print("Accuracy (raw pixels):", accuracy_score(y_test, y_pred_raw))

cm_raw = confusion_matrix(y_test, y_pred_raw)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_raw, display_labels=labels)
disp.plot(cmap="Blues", values_format="d")
plt.title("MNIST Confusion Matrix - raw pixels")
plt.show()
